# ATLAS Group Notebook: Finding the Z Boson with Muons
## Dimuon Invariant Mass and the Muon Spectrometer

**Vanderbilt Programs for Talented Youth | Instructor: Jennifer James, Ph.D. Candidate**

---

### The challenge

Muons are heavier cousins of the electron, and they behave very differently in a detector. Electrons and photons shower and get absorbed in the calorimeters. Muons punch straight through, barely losing energy along the way (they are called minimum-ionizing particles). That is exactly why ATLAS puts a dedicated **muon spectrometer** as its outermost layer: it is the only subsystem a muon reaches at all.

In this notebook, you will reconstruct the **Z boson** using its decay to two muons, Z to mu+ mu-. This is one of the cleanest, most reliable calibration channels in all of particle physics; ATLAS and CMS use it constantly to check that their muon systems are working correctly.

---

$$m_{\mu\mu}^2 = (E_1 + E_2)^2 - |\vec{p}_1 + \vec{p}_2|^2$$

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(91)  # a nod to the Z boson mass, ~91 GeV
print("Libraries loaded!")

---

## Part 1: Generating the Dimuon Mass Spectrum

We simulate the dimuon invariant mass spectrum as ATLAS would see it:

- **Background**: a falling continuum from Drell-Yan production of muon pairs at random combinations of momentum (q + qbar to gamma* to mu+ mu-, with no Z boson involved) plus combinatorial pairs from unrelated muons in the same event
- **Signal**: a peak at 91.19 GeV from real Z to mu+ mu- decays. The width of this peak is set by the **muon spectrometer's momentum resolution**, not by a calorimeter

In [ ]:
# ── Mass range and binning ─────────────────────────────────────────────────────────────
m_bins    = np.linspace(40, 140, 101)   # 1 GeV bins, 40-140 GeV
m_centers = 0.5 * (m_bins[:-1] + m_bins[1:])
m_Z       = 91.19   # GeV/c2 (PDG value)

# ── Background model: falling Drell-Yan and combinatorial continuum ────────────
def background(m, N_bkg, k):
    """Falling power-law-like continuum, typical of Drell-Yan below the Z peak."""
    return N_bkg * m**(-k)

N_bkg  = 60000
k_true = 3.0

bkg_rate = background(m_centers, N_bkg, k_true)
bkg_rate = bkg_rate / bkg_rate.sum() * 4000   # normalize to a realistic total background count

# ── Signal model: Gaussian at the Z mass ────────────────────────────────────
N_signal    = 2500   # Z to mumu events (Z production is common, unlike Higgs)
sigma_muon  = 2.3    # GeV, muon spectrometer momentum resolution propagated to mass

sig_rate = N_signal * np.exp(-0.5 * ((m_centers - m_Z)/sigma_muon)**2)
sig_rate /= sig_rate.sum()
sig_rate *= N_signal

# ── Generate observed data (Poisson fluctuations) ─────────────────────────────
total_rate   = bkg_rate + sig_rate
observed     = np.random.poisson(total_rate)
observed_err = np.sqrt(np.maximum(observed, 1))

print(f"Simulated dimuon mass spectrum:")
print(f"  Total events:    {observed.sum():,}")
print(f"  Background:      ~{int(bkg_rate.sum()):,}")
print(f"  Signal (Z boson): ~{N_signal} events at {m_Z:.2f} GeV")
print(f"  Muon spectrometer mass resolution: sigma = {sigma_muon} GeV")

---

## Part 2: The Raw Spectrum: Can You See the Z Boson?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(m_centers, observed, yerr=observed_err,
            fmt='o', color='black', markersize=4, capsize=2,
            linewidth=1, label='Data (observed)')
ax.plot(m_centers, bkg_rate + sig_rate, color='firebrick',
        linewidth=2, label='True signal + background (hidden in real life!)', alpha=0.6)
ax.set_xlabel(r'Dimuon invariant mass $M_{\mu\mu}$ (GeV/c$^2$)', fontsize=12)
ax.set_ylabel('Events / 1 GeV', fontsize=12)
ax.set_title('Raw dimuon mass spectrum\nThe Z boson peak should already be visible here', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(40, 140)
ax.axvline(x=m_Z, color='gold', linestyle='--', linewidth=1.5, alpha=0.6)
plt.tight_layout()
plt.show()

print("Unlike the Higgs, the Z boson is produced often enough that its peak stands out")
print("even in the raw spectrum. This is exactly why Z to mumu is used as a calibration")
print("channel: physicists know precisely where the peak should be, so any shift or")
print("broadening tells them something about the detector, not about new physics.")

---

## Part 3: Fitting the Peak

Even though the Z peak is visible by eye, we still fit it properly: a background fit using the sideband regions (away from the peak), subtracted from the data, followed by a Gaussian fit to what remains. This mirrors the same sideband-subtraction method used for the Higgs searches.

In [ ]:
sideband_mask = (m_centers < 81) | (m_centers > 101)
m_fit   = m_centers[sideband_mask]
obs_fit = observed[sideband_mask]
err_fit = observed_err[sideband_mask]

popt_bkg, pcov_bkg = curve_fit(background, m_fit, obs_fit, p0=[N_bkg, k_true],
                                sigma=err_fit, absolute_sigma=True, maxfev=10000)

bkg_fitted = background(m_centers, *popt_bkg)
residuals  = observed - bkg_fitted

signal_mask = (m_centers >= 81) & (m_centers <= 101)
m_sig   = m_centers[signal_mask]
res_sig = residuals[signal_mask]
err_sig = observed_err[signal_mask]

def gaussian(m, N, mu, sigma):
    return N * np.exp(-0.5*((m - mu)/sigma)**2)

popt_sig, pcov_sig = curve_fit(gaussian, m_sig, res_sig,
                                p0=[N_signal, m_Z, sigma_muon], sigma=err_sig,
                                absolute_sigma=True, maxfev=10000)
perr_sig = np.sqrt(np.diag(pcov_sig))
N_sig_fit, mu_fit, sigma_fit = popt_sig
N_sig_err, mu_err, sigma_err = perr_sig

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.errorbar(m_centers, observed, yerr=observed_err,
            fmt='o', color='black', markersize=4, capsize=2, label='Data')
ax.plot(m_centers, bkg_fitted, color='steelblue', linewidth=2, label='Background fit (sidebands)')
ax.axvspan(81, 101, alpha=0.08, color='gold', label='Signal region (excluded from bkg fit)')
ax.axvline(x=m_Z, color='gold', linestyle=':', linewidth=1.5)
ax.set_xlabel(r'$M_{\mu\mu}$ (GeV/c$^2$)', fontsize=12)
ax.set_ylabel('Events / 1 GeV', fontsize=12)
ax.set_title('Background fit using sidebands', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.errorbar(m_sig, res_sig, yerr=err_sig,
            fmt='o', color='black', markersize=6, capsize=3, label='Residuals (data - bkg)')
m_fine = np.linspace(81, 101, 200)
ax.plot(m_fine, gaussian(m_fine, *popt_sig), color='firebrick', linewidth=2.5, label='Gaussian fit')
ax.fill_between(m_fine, 0, gaussian(m_fine, *popt_sig), alpha=0.2, color='firebrick')
ax.axvline(x=m_Z, color='gold', linestyle=':', linewidth=1.5, label=f'PDG Z mass: {m_Z} GeV')
ax.set_xlabel(r'$M_{\mu\mu}$ (GeV/c$^2$)', fontsize=12)
ax.set_ylabel('Events / 1 GeV (bkg subtracted)', fontsize=12)
ax.set_title('The Z boson signal', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Fitted Z mass:              {mu_fit:.2f} +/- {mu_err:.2f} GeV/c2")
print(f"Fitted mass resolution:     {abs(sigma_fit):.2f} +/- {sigma_err:.2f} GeV")
print(f"Fitted signal yield:        {N_sig_fit:.0f} +/- {N_sig_err:.0f} events")
print()
print(f"True PDG Z mass: {m_Z} GeV/c2")
print(f"Difference: {abs(mu_fit - m_Z):.3f} GeV ({abs(mu_fit-m_Z)/m_Z*100:.3f}%)")

---

## Part 4: Muon Momentum and the Curvature Formula

Muon momentum, like all charged-particle momentum in a magnetic field, comes from track curvature:

$$p = qBr$$

ATLAS actually measures muon momentum **twice**: once in the inner tracker (small radius, close to the collision point) and once in the outer muon spectrometer (much larger radius, its own separate magnet system). Combining both measurements gives better precision than either alone, especially at high momentum, where the inner tracker's short lever arm makes tracks look almost straight.

In [ ]:
# ── Toy comparison: inner tracker alone vs. muon spectrometer alone vs. combined ──
pT_values = np.linspace(5, 200, 100)   # GeV/c, transverse momentum range

def resolution_inner(pT):
    """Inner tracker: short lever arm, resolution degrades quickly at high pT."""
    return 0.02 + 0.0015 * pT

def resolution_muon_spec(pT):
    """Muon spectrometer: long lever arm, better at high pT, worse at low pT
    due to multiple scattering in material."""
    return 0.05 + 0.0004 * pT

def resolution_combined(pT):
    """Combining both measurements (added in quadrature, inverse-variance weighted)."""
    r1 = resolution_inner(pT)
    r2 = resolution_muon_spec(pT)
    return 1.0 / np.sqrt(1/r1**2 + 1/r2**2)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(pT_values, resolution_inner(pT_values)*100, color='steelblue',
        linewidth=2, label='Inner tracker alone')
ax.plot(pT_values, resolution_muon_spec(pT_values)*100, color='darkorange',
        linewidth=2, label='Muon spectrometer alone')
ax.plot(pT_values, resolution_combined(pT_values)*100, color='firebrick',
        linewidth=2.5, label='Combined (inner + muon spectrometer)')
ax.set_xlabel(r'Muon $p_T$ (GeV/c)', fontsize=12)
ax.set_ylabel('Relative momentum resolution (%)', fontsize=12)
ax.set_title('Why ATLAS combines two independent momentum measurements for muons', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("At low pT, the inner tracker wins (short path, less multiple scattering).")
print("At high pT, the muon spectrometer wins (its large radius means more curvature")
print("to measure, even though the momentum itself makes the track straighter).")
print("Combining both gives the best resolution across the full range.")

---

## Your Group's Checkpoint Questions

1. In Part 2, the Z peak was already visible in the raw spectrum, unlike the Higgs peak your classmates worked with. Why is a Z boson so much easier to see than a Higgs boson? What does this have to do with production rate?

2. Why do muons reach the muon spectrometer at all, when electrons and photons are absorbed by the calorimeters long before reaching that far out?

3. In Part 4, why does a longer lever arm (bigger radius) help more at high momentum specifically? Think about what "more curvature to measure" means for a nearly straight, high-momentum track.

4. If your detector design had to choose between a bigger inner tracker or a bigger muon spectrometer, but not both, which would you prioritize if your physics goal was measuring very high-momentum muons (for example, from a hypothetical new heavy particle)? Why?

5. From your subsystem checklist, does this notebook change how you would prioritize your detector's magnet system? Specifically, does the muon spectrometer need its own separate magnet, or could it share the same magnet as the inner tracker?